# Merchant Fraud Risk Profile

This notebook covers **Member 4 fraud risk only**. It does not calculate a business score, tune ranking weights, or produce a Top 100. The primary output combines consumer exposure with a cross-validated KNN merchant-risk score.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

def find_member4_dir(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        direct = candidate if candidate.name == 'member4_fraud' else candidate / 'member4_fraud'
        if (direct / 'code').is_dir():
            return direct
    raise FileNotFoundError('Run this notebook from the repository or member4_fraud/code.')

MEMBER4_DIR = find_member4_dir()
CODE_DIR = MEMBER4_DIR / 'code'
sys.path.insert(0, str(CODE_DIR))

from build_fraud_risk_profile import build_fraud_risk_profile

OUTPUT_DIR = MEMBER4_DIR / 'result'
summary = build_fraud_risk_profile(output_dir=OUTPUT_DIR)
display(summary)

## 1. Consumer risk exposure

For merchant $m$, consumer exposure is the amount-weighted average consumer risk:

$$C_m = \frac{\sum_{t \in T_m^*} Amount_t \cdot ConsumerRisk_{u(t)}}{\sum_{t \in T_m^*} Amount_t}$$

It is converted to a merchant percentile $C_m^*$. Transaction and amount coverage are retained as reliability indicators but are not included in the score formula.

In [ ]:
profile = pd.read_csv(OUTPUT_DIR / 'merchant_fraud_risk_profile.csv')
status_counts = profile['fraud_evidence_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='merchants')
coverage_columns = [
    'consumer_risk_transaction_coverage', 'consumer_risk_amount_coverage',
    'observed_consumer_label_transaction_coverage',
    'observed_consumer_label_amount_coverage'
]
coverage_summary = profile[coverage_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
display(status_counts, coverage_summary)

## 2. KNN merchant-risk score

The 61 directly observed merchants provide the KNN training targets. After median imputation and StandardScaler, repeated cross-validation chooses K and neighbour weighting. The fitted KNN scores all eligible merchants, and its relative signal is:

$$K_m = Percentile(KNNPredictedMerchantFraud_m)$$

Direct observation count, mean and maximum fraud probability remain diagnostics and training evidence only. They are not inserted directly into the final formula.

In [ ]:
knn_columns = [
    'merchant_abn', 'merchant_name', 'knn_score',
    'knn_merchant_risk_percentile', 'fraud_risk_index',
    'risk_safety_score'
]
knn_profile = profile.loc[profile.has_knn_merchant_risk_information, knn_columns]
display(knn_profile.sort_values('knn_merchant_risk_percentile', ascending=False).head(10))

## 3. Combined fraud risk

The baseline gives equal weight to the consumer-exposure percentile and KNN merchant-risk percentile:

$$FraudRisk_m = \begin{cases}0.5C_m^* + 0.5K_m, & C,K\text{ both available}\\C_m^*, & C\text{ only}\\K_m, & K\text{ only}\end{cases}$$

KNN covers all eligible merchants; eight merchants without consumer exposure therefore use the KNN percentile alone. No merchant ranking is calculated here.

In [ ]:
knn_scores = pd.read_csv(OUTPUT_DIR / 'knn_fraud_risk_scores.csv')
display(knn_scores.describe().T, knn_scores.head())

## 4. Interpretation and limitations

- `fraud_risk_index` is a relative risk index, **not** a fraud probability.
- Only 61 merchants have direct merchant fraud observations.
- The KNN merchant score is learned from only 61 directly observed merchants, so uncertainty remains high.
- Consumer exposure uses model-estimated consumer risk and is not proof of merchant wrongdoing.
- KNN model-score coverage and raw consumer-label coverage are reported separately; the former can be complete even when direct label evidence is sparse.
- Coverage, direct observation count, and maximum observed risk must accompany the index as reliability diagnostics.
- Eight merchants have KNN-only risk because consumer exposure is unavailable; no merchant is assigned zero merely because evidence is missing.